In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBRegressor
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import requests
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
#pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="muted")

In [4]:
DATA_DIR = Path("Data")
END_DATE = "2026-01-31"
DELAY_THRESHOLD = 5 

In [5]:
# notice that the 2022-2024 data are xlsx files while 2025+ is a csv file 
xlsx_sources = [
    (DATA_DIR/"ttc-subway-delay-data-2022.xlsx", "2022"),
    (DATA_DIR/"ttc-subway-delay-data-2023.xlsx", "2023"),
    (DATA_DIR/"ttc-subway-delay-data-2024.xlsx", "Subway"),
]
dfs = [pd.read_excel(p, sheet_name=s) for p, s in xlsx_sources]

# 2025+ csv has an extra _id column so we must drop it
df_2025 = pd.read_csv(DATA_DIR / "TTC Subway Delay Data since 2025.csv")
df_2025 = df_2025.drop(columns=["_id"], errors="ignore")
dfs.append(df_2025)

df = pd.concat(dfs, ignore_index=True)
print(f"Combined shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range : {pd.to_datetime(df['Date']).min().date()} to {pd.to_datetime(df['Date']).max().date()}")
df.head(10)

Combined shape : 97,502 rows x 10 columns
Date range : 2022-01-01 to 2026-01-31


,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,2022-01-01 00:00:00,15:59,Saturday,LAWRENCE EAST STATION,SRDP,0,0,N,SRT,3023
1,2022-01-01 00:00:00,02:23,Saturday,SPADINA BD STATION,MUIS,0,0,NaN,BD,0
2,2022-01-01 00:00:00,22:00,Saturday,KENNEDY SRT STATION TO,MRO,0,0,NaN,SRT,0
3,2022-01-01 00:00:00,02:28,Saturday,VAUGHAN MC STATION,MUIS,0,0,NaN,YU,0
4,2022-01-01 00:00:00,02:34,Saturday,EGLINTON STATION,MUATC,0,0,S,YU,5981
5,2022-01-01 00:00:00,05:40,Saturday,QUEEN STATION,MUNCA,0,0,NaN,YU,0
6,2022-01-01 00:00:00,06:56,Saturday,DAVISVILLE STATION,MUNCA,0,0,NaN,YU,0
7,2022-01-01 00:00:00,06:58,Saturday,ST PATRICK STATION,MUNCA,0,0,NaN,YU,0
8,2022-01-01 00:00:00,07:01,Saturday,PAPE STATION,MUNCA,0,0,NaN,BD,0
9,2022-01-01 00:00:00,07:43,Saturday,WILSON STATION,TUATC,10,0,S,YU,5896


In [9]:
info_df = pd.DataFrame({
    "dtype"     : df.dtypes,
    "non_null"  : df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_%"    : (df.isna().mean() * 100).round(1),
})
print(info_df.to_string())
print("Min Delay Statistics:")
print(df["Min Delay"].describe().round(2).to_string())

            dtype  non_null  null_count  null_%
Date       object     97502           0     0.0
Time          str     97502           0     0.0
Day           str     97502           0     0.0
Station       str     97502           0     0.0
Code          str     97502           0     0.0
Min Delay   int64     97502           0     0.0
Min Gap     int64     97502           0     0.0
Bound         str     63680       33822    34.7
Line          str     97296         206     0.2
Vehicle     int64     97502           0     0.0
Min Delay Statistics:
count    97502.00
mean         3.06
std         11.90
min          0.00
25%          0.00
50%          0.00
75%          4.00
max        900.00
